<a href="https://colab.research.google.com/github/OPA-kan/lispsxnll/blob/main/Project_GGG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Graph-Guided Generation**
**グラフ理論を用いてAIを制御する**

目的:なぜ魅力的な文章が書けるかをホワイトボックスにして汎用性を上げるため

使うマーケティングの理論と心理学の理論は3つ

1.   Means-End Chain Theory
2.   意味ネットワーク
3.   Sensory Marketing


マーケティング理論をグラフ化して落とし込み、意味ネットワークで補強分析し、それを基にAIで制御することでPR文の最適な生成を目指す

ちな4本といいつつ、ほぼ松下しか使ってない

松下（目的）→ Grunert（構造）→ 日高（探索）→ Celis（品質管理）

Perceptual Input (知覚入力): VLMによる画像解析

Source: Image FeaturesSemantic Processing (意味処理): 松下モデルによる意味変換

Transformation: Attribute $\to$ ConsequenceValue Integration (価値統合): 手段目標連鎖による高次価値への接続

Transformation: Consequence $\to$ ValueGenerative Output (生成出力): グラフパスに基づくコピーライティング

In [1]:
from pathlib import Path
PROJECT_ROOT = Path("/content/drive/MyDrive/NISHIKA_AI_Competition") #@param {type:"string"}
# Google Driveをマウント
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 解凍
!pip install polars
import os
import polars as pl
import zipfile
from PIL import Image
from pathlib import Path

# パス設定
INPUT_DIR = PROJECT_ROOT / "input"
OUTPUT_DIR = PROJECT_ROOT / "output"
MODEL_DIR = INPUT_DIR / "model"
TRAIN_CSV = INPUT_DIR / 'train.csv'
TEST_CSV = INPUT_DIR / 'test.csv'
IMAGES_ZIP = INPUT_DIR / 'images.zip'
TEMP_IMAGE_DIR = Path('/content/extracted')  # 画像ファイルの一時展開先

IMAGES_DIR = TEMP_IMAGE_DIR / 'images'
POOLING_DIR = MODEL_DIR / '1_Pooling'
POOLING_CONFIG_PATH = POOLING_DIR / 'config.json'

# データ読み込み
try:
    # n_rows=100 を追加することで、読み込む行数を先頭から100行に制限中
    # 学習だけ制限解除なう
    # 提出モードなうーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーーー
    # train_df = pl.read_csv(TRAIN_CSV, n_rows=100)
    # test_df = pl.read_csv(TEST_CSV, n_rows=100)
    train_df = pl.read_csv(TRAIN_CSV)
    test_df = pl.read_csv(TEST_CSV)
    display(train_df[0:10])
    display(test_df[0:10])
except FileNotFoundError:
    print(f"エラー: {TRAIN_CSV} または {TEST_CSV} が見つかりません。Google Driveがマウントされているか、パスが正しいか確認してください。")
    print(f"現在のPROJECT_ROOT: {PROJECT_ROOT}")
    # 代替処理やエラーハンドリングをここに追加できます
    # 例: 空のDataFrameを作成するなど
    train_df = pl.DataFrame()
    test_df = pl.DataFrame()
if not TEMP_IMAGE_DIR.exists():
    print(f"📂 画像を解凍中...: {IMAGES_ZIP}")
    with zipfile.ZipFile(IMAGES_ZIP, 'r') as zip_ref:
        zip_ref.extractall(TEMP_IMAGE_DIR)
    print("✅ 解凍完了！")
else:
    print("ℹ️ 画像は既に解凍済みです。")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.4/802.4 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 MB 82.3 MB/s eta 0:00:00


ID,label
str,str
"""00u3zLb5""","""薄衣のイタリア風フライでふわっサクッの食感をお楽しみください"""
"""016u6IXS""","""欧州製造のクリームチーズに爽やかな４種のハーブを合わせました…"
"""01gZaTt9""","""心地よいなめらかな肌触りで、どなたでも簡単に本格的なかっさフ…"
"""01w8O1fU""","""食べると口の中が青くなるぶどう味のガムです。"""
"""06012zbE""","""99%カットフィルタで花粉・細菌・ウイルス・PM2.5等の侵…"
"""07SkAvVc""","""大人のエチケット！刃が肌に当たりにくい安心設計！簡単！まわす…"
"""08Dir9VO""","""鹿児島県産の桜島小みかん果汁を配合した爽やかな香りと濃厚な甘…"
"""09549FFh""","""3個に1個超すっぱい！ゲーム感覚で楽しめるソーダ味のやわらか…"
"""09n7N0SI""","""桜のチップで燻製しました。ほんのりした甘さと酸味が特徴のいか…"


ID
str
"""06j3BNM0"""
"""09n0l3iy"""
"""0B4XRHmr"""
"""0CIltEix"""
"""0CupiBOe"""
"""0Dj30YmI"""
"""0GM19aJi"""
"""0HrBR5Hi"""
"""0NonR9si"""


📂 画像を解凍中...: /content/drive/MyDrive/NISHIKA_AI_Competition/input/images.zip
✅ 解凍完了！


構造化OCR(1枚目メイン、2枚目はサブ)

In [ ]:
# OCR
import os
import io
import polars as pl
from tqdm import tqdm
from google.cloud import vision

# ---------------------------------------------------------
# 設定 & 準備
# ---------------------------------------------------------
# 認証
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(PROJECT_ROOT / "service_account.json")
OCR_SAVE_PATH = OUTPUT_DIR / "ocr_results_google_structured.csv"

# ---------------------------------------------------------
# 関数: Google Vision API (単体)
# ---------------------------------------------------------
def detect_text_google(path):
    """Google Vision APIで画像をテキスト化"""
    client = vision.ImageAnnotatorClient()
    try:
        with io.open(path, 'rb') as image_file:
            content = image_file.read()
        image = vision.Image(content=content)

        # TEXT_DETECTION (一般文字) または DOCUMENT_TEXT_DETECTION (文書)
        # パッケージはデザイン文字が多いので TEXT_DETECTION が無難
        response = client.text_detection(image=image)
        texts = response.text_annotations

        if texts:
            # texts[0]に全文が入っている
            return texts[0].description
    except Exception as e:
        # print(f"Error: {e}") # エラーは握りつぶして続行
        pass
    return ""

# ---------------------------------------------------------
# 関数: 構造化ロジック (ここがキモ)
# ---------------------------------------------------------
def get_structured_ocr_google(product_id, images_dir, train_test):
    # 画像パス取得
    product_dir = os.path.join(images_dir, train_test ,str(product_id))
    if not os.path.exists(product_dir): return ""

    # 1.jpg, 2.jpg... の順にソート
    files = sorted([f for f in os.listdir(product_dir) if f.endswith(".jpg")])
    image_paths = [os.path.join(product_dir, f) for f in files]
    if not image_paths: return ""

    structured_text = ""

    # --- 1枚目 (メイン画像) ---
    # Google先生の本気を見る
    text_main = detect_text_google(image_paths[0])
    if text_main:
        # 改行をスペースに置換して整形
        text_main = text_main.replace("\n", " ")
        structured_text += f"【パッケージ正面の情報(Google Vision)】\n{text_main}\n"

    # --- 2枚目以降 (補足情報) ---
    if len(image_paths) > 1:
        structured_text += "\n【裏面・補足情報】\n"
        for sub_path in image_paths[1:]:
            text_sub = detect_text_google(sub_path)
            if text_sub:
                text_sub = text_sub.replace("\n", " ")
                structured_text += f"{text_sub} "
                ocr_text = structured_text

    return structured_text

# ---------------------------------------------------------
# メイン処理
# ---------------------------------------------------------
print("Phase 1: Google Vision API Structured OCR...")

# キャッシュ確認
if os.path.exists(OCR_SAVE_PATH):
    print(f"📂 保存済みのGoogle OCR結果を読み込みます: {OCR_SAVE_PATH}")
    ocr_df = pl.read_csv(OCR_SAVE_PATH)

else:
    print("⚡️ 新規にGoogle Vision APIを実行します（課金発生）...")
    ocr_results = []

    # テストデータ全件処理
    for idx in tqdm(range(len(test_df))):
        product_id = test_df[idx, "ID"]

        # 構造化テキストを取得
        txt = get_structured_ocr_google(product_id, IMAGES_DIR, "test")

        ocr_results.append({"ID": product_id, "OCR_TEXT": txt})

    ocr_df = pl.DataFrame(ocr_results)
    ocr_df.write_csv(OCR_SAVE_PATH)
    print(f"💾 保存完了: {OCR_SAVE_PATH}")

# 確認
print(f"データ数: {len(ocr_df)}")
print("\n【サンプル表示】")
print(ocr_df["OCR_TEXT"][0])

### **HVM構築(phase1)**
属性抽出から抽出したChainを統合し、ノード間の結合強度（重み）を持つHVM（有向グラフ）を構築する
Means-End Chain Theory (Grunert):

効果: 「属性・機能・価値」というタグ付けをしてデータを整理した。これのおかげで、AIは「美味しい（機能）」と「幸せ（価値）」を区別する。

In [ ]:
!pip install google.generativeai
import pandas as pd
from tqdm import tqdm
import json
import time
import google.generativeai as genai
from google.colab import userdata
from pathlib import Path

# ==========================================
# 1. 設定 & Gemini初期化
# ==========================================
PROJECT_ROOT = Path('/content/drive/MyDrive/NISHIKA_AI_Competition')
OUTPUT_DIR = PROJECT_ROOT / "output"
TRAIN_CSV = PROJECT_ROOT / "input" / "train.csv"

# APIキー設定 (ColabのSecrets機能推奨)
try:
    GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
except:
    GOOGLE_API_KEY = "YOUR_NEW_AND_VALID_API_KEY_HERE" # ★ここに新しいAPIキーを貼り付けてください

genai.configure(api_key=GOOGLE_API_KEY)

MODEL_NAME = 'gemini-2.0-flash'
model = genai.GenerativeModel(MODEL_NAME)

print(f"🤖 Model: {MODEL_NAME} を使用してマイニングを開始します。")

# ==========================================
# 2. データ準備
# ==========================================
df_train = pd.read_csv(TRAIN_CSV)

# 学習に使いやすい「中くらいの長さ(40-90文字)」の良質な正解データを選ぶ(テスト用)
# df_mining = df_train[(df_train['label'].str.len() > 40) & (df_train['label'].str.len() < 90)].sample(50, random_state=42).reset_index(drop=True)
# 全データマイニング
df_mining = df_train
print(f"⛏️ マイニング対象: {len(df_mining)} 件")

# ==========================================
# 3. Gemini Mining Class
# ==========================================
class GeminiMiner:
    def __init__(self, model):
        self.model = model

    def extract_logic(self, pr_text):
        # JSONモードを強制するプロンプト
        prompt = f"""あなたはマーケティング分析官です。
以下の「優れたPR文」を分析し、Means-End Chain理論に基づいて構成要素を抽出してください。

【対象PR文】
{pr_text}

【タスク】
このPR文が「なぜ売れるのか」を分解し、以下のJSONフォーマットで出力してください。
※該当がない項目は null にしてください。
※「属性」は商品の事実、「便益」は機能的メリット、「価値」は情緒的ゴールです。

出力形式(JSON):
{{
  "Attribute": "抽出した属性(成分・素材など)",
  "Sensory": "抽出した感覚表現(オノマトペ・シズル感)",
  "Benefit": "抽出した便益(時短・使いやすさなど)",
  "Value": "抽出した価値(癒やし・安心感など)",
  "Reasoning": "このPR文が優れている理由を一言で"
}}
"""
        try:
            response = self.model.generate_content(
                prompt,
                generation_config={"response_mime_type": "application/json"}
            )
            return json.loads(response.text)
        except Exception as e:
            print(f"Error: {e}")
            return None

# ==========================================
# 4. 実行 (Mining Execution)
# ==========================================
miner = GeminiMiner(model)
extracted_data = []

print("🚀 Geminiがロジックを抽出中...")

for i in tqdm(range(len(df_mining))):
    target_text = df_mining.loc[i, 'label']

    # API制限対策 (少し待機)
    time.sleep(1.5)

    logic_json = miner.extract_logic(target_text)

    if logic_json:
        # Ensure logic_json is a dictionary even if the model returns a list containing one
        if isinstance(logic_json, list) and len(logic_json) > 0:
            logic_json = logic_json[0]

        # 元のテキストも紐付けておく
        logic_json["Original_PR"] = target_text
        extracted_data.append(logic_json)

# ==========================================
# 5. 保存 (Knowledge Graph)
# ==========================================
df_knowledge = pd.DataFrame(extracted_data)

# 結果を見やすく並べ替え
cols = ["Original_PR", "Attribute", "Sensory", "Benefit", "Value", "Reasoning"]

# extracted_dataが空の場合にKeyErrorが発生しないようにチェックを追加
if not df_knowledge.empty and all(col in df_knowledge.columns for col in cols):
    df_knowledge = df_knowledge[cols]
else:
    print("Warning: df_knowledge is empty or missing expected columns. Skipping column reordering.")

SAVE_PATH = OUTPUT_DIR / "marketing_knowledge_graph_gemini.csv"
df_knowledge.to_csv(SAVE_PATH, index=False)

print(f"\n✅ マイニング完了！")
print(f"知見データベースを保存しました: {SAVE_PATH}")

# サンプル表示
print("\n【Geminiが発見したロジック (Top 3)】")
pd.set_option('display.max_colwidth', None)
try:
    display(df_knowledge.head(3))
except:
    print(df_knowledge.head(3))


🤖 Model: gemini-2.0-flash を使用してマイニングを開始します。
⛏️ マイニング対象: 4560 件
🚀 Geminiがロジックを抽出中...


 15%|█▍        | 662/4560 [33:50<3:25:21,  3.16s/it]

### **推論(phase2)**

知覚入力(OCR&VLM)
Sensory Marketing:

松下モデルの「属性（Attributes）」に、単なるスペックだけでなく「とろ〜り」「サクサク」などの感覚情報を注入。

In [1]:
# ==========================================
# 0. 準備: ライブラリのアップデート (最新版)
# ==========================================
# 実行後に「RESTART SESSION」ボタンが出たら押してください
%%capture
!pip install unsloth
# Colabの環境に合わせて最新の依存関係を解決してくれるタグ
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
import torch
from unsloth import FastVisionModel
from PIL import Image
from tqdm import tqdm
import pandas as pd
import os
from pathlib import Path

# ==========================================
# 1. 設定 & データ読み込み
# ==========================================
PROJECT_ROOT = Path('/content/drive/MyDrive/NISHIKA_AI_Competition')
INPUT_DIR = PROJECT_ROOT / "input"
OUTPUT_DIR = PROJECT_ROOT / "output"
# 画像フォルダ (testフォルダを指定)
IMAGES_DIR = Path('/content/extracted/images/test')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# テストデータ (IDリスト)
df_test = pd.read_csv(INPUT_DIR / "test.csv")

# OCRデータ (すでに作成済みの構造化データ)
OCR_FILE = OUTPUT_DIR / "ocr_results_google_structured.csv"
if not OCR_FILE.exists():
    raise FileNotFoundError("OCRデータが見つかりません！ocr_results_google_structured.csv を用意してください。")

df_ocr = pd.read_csv(OCR_FILE)

# ★ここが重要: ID順を保証しつつ、OCRデータもくっつけておく
df_input = df_test[['ID']].merge(df_ocr[['ID', 'OCR_TEXT']], on='ID', how='left')
print(f"処理対象: {len(df_input)} 件")

# ==========================================
# 2. モデルロード (Llama-3.2-11B-Vision)
# ==========================================
print("👁️ VLM (Llama-3.2-11B-Vision) をロード中...")
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit",
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth",
)
FastVisionModel.for_inference(model)

# ==========================================
# 3. Sensory Extraction Logic (Best Prompt)
# ==========================================
def extract_sensory_features(image_path):
    if not os.path.exists(image_path): return None

    image = Image.open(image_path).convert("RGB")

    # ★コード(1)の「構造化プロンプト」を採用（後で使いやすい）
    instruction = """
    You are a professional copywriter. Analyze this product image and extract "Sensory Keywords" that make people want to buy it.

    Target:
    1. **Color/Vibe**: (e.g., Warm Orange, Retro, Luxury Gold)
    2. **Texture/Sizzle**: (e.g., Crispy, Fluffy, Juicy, Glossy, Thick)
    3. **Visual Impact**: (e.g., Voluminous, Cute, Elegant)

    CONSTRAINT: Output strictly in Japanese. Use emotional adjectives.

    Output Format Example:
    [色彩: 温かみのあるオレンジ], [食感: 外はカリッ中はトロッ], [見た目: 高級感あふれるパッケージ]
    """

    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": instruction}
        ]}
    ]

    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens = False,
        return_tensors = "pt",
    ).to("cuda")

    # 生成 (Temperature 0.6 で少し表現豊かに)
    outputs = model.generate(
        **inputs,
        max_new_tokens = 100,
        use_cache = True,
        temperature = 0.6,
        do_sample = True
    )

    result = tokenizer.batch_decode(outputs)[0].split("<|start_header_id|>assistant<|end_header_id|>\n\n")[-1]
    return result.replace("<|eot_id|>", "").strip()

# ==========================================
# 4. 実行 (Execution)
# ==========================================
print("🚀 画像から『シズル感』を抽出しています...")

sensory_results = []

# 全件処理
for i in tqdm(range(len(df_input))):
    pid = str(df_input.loc[i, 'ID'])

    # 画像パス検索
    product_dir = IMAGES_DIR / pid
    img_path = None

    if product_dir.exists():
        files = list(product_dir.glob("*.jpg"))
        if files:
            img_path = files[0] # 1枚目を使用

    sensory_text = ""
    if img_path:
        try:
            sensory_text = extract_sensory_features(img_path)
        except Exception as e:
            pass # エラー時は空文字

    sensory_results.append(sensory_text)

# データフレームに統合
df_input['Sensory_Features'] = sensory_results

# ==========================================
# 5. 保存
# ==========================================
# OCRデータもVLMデータも入った入力データが完成
SAVE_PATH = OUTPUT_DIR / "test_input_features_vlm_structured.csv"
df_input.to_csv(SAVE_PATH, index=False)

print(f"\n✅ 完了！")
print(f"グラフ生成用の統合データを保存しました: {SAVE_PATH}")
print("\n【抽出結果サンプル】")
display(df_input[['ID', 'OCR_TEXT', 'Sensory_Features']].head())

処理対象: 1190 件
👁️ VLM (Llama-3.2-11B-Vision) をロード中...
==((====))==  Unsloth 2025.12.5: Fast Mllama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🚀 画像から『シズル感』を抽出しています...


100%|██████████| 1190/1190 [00:00<00:00, 38385.45it/s]


✅ 完了！
グラフ生成用の統合データを保存しました: /content/drive/MyDrive/NISHIKA_AI_Competition/output/test_input_features_vlm_structured.csv

【抽出結果サンプル】


,ID,OCR_TEXT,Sensory_Features
0,06j3BNM0,【パッケージ正面の情報(Google Vision)】\nはじめてのおじまじと mizkan...,
1,09n0l3iy,【パッケージ正面の情報(Google Vision)】\nAsian アジアン ・紀行と グ...,
2,0B4XRHmr,【パッケージ正面の情報(Google Vision)】\n岩埕烈荣 香ばしさ アップ (深煎...,
3,0CIltEix,【パッケージ正面の情報(Google Vision)】\nCOOL MINTの香り 不布マス...,
4,0CupiBOe,【パッケージ正面の情報(Google Vision)】\nNISSIN 88533 00 C...,


### **GGG-Brain**
意味経路探索(HVM上とOCR/VLMを探索し、Start NodeからTarget Value（訴求したい価値）のエッジに至るパスを検索する。)

「食塩」という属性から「安心感」という価値へどうジャンプするか、その「線路」は日高理論が元

In [ ]:
import networkx as nx
import pandas as pd
import ast
from tqdm import tqdm
from pathlib import Path

# ==========================================
# 1. 設定 & データ読み込み
# ==========================================
PROJECT_ROOT = Path('/content/drive/MyDrive/NISHIKA_AI_Competition')
OUTPUT_DIR = PROJECT_ROOT / "output"

# 入力　HVMの脳みそ
KNOWLEDGE_GRAPH_CSV = OUTPUT_DIR / "marketing_knowledge_graph_gemini.csv"
# 入力　入力データ
INPUT_FEATURES_CSV = OUTPUT_DIR / "test_input_features_vlm_structured.csv"

df_input = pd.read_csv(INPUT_FEATURES_CSV)
df_kg = pd.read_csv(KNOWLEDGE_GRAPH_CSV)

# ==========================================
# 2. HVM (Graph) の構築
# ==========================================
print("🧠 HVM (Brain) を構築中...")

G = nx.DiGraph()
node_types = {} # ノードの種類を記録 (Attribute, Sensory, Benefit, Value)

def clean_split(text):
    if pd.isna(text): return []
    text = str(text).replace("、", ",").replace("。", "")
    return [x.strip() for x in text.split(",") if x.strip()]

# グラフのエッジを張る
for _, row in df_kg.iterrows():
    # 各層のノードを取得
    attrs = clean_split(row['Attribute'])
    senses = clean_split(row['Sensory'])
    benefits = clean_split(row['Benefit'])
    values = clean_split(row['Value'])

    # ノード属性を記録
    for n in attrs: node_types[n] = 'Attribute'
    for n in senses: node_types[n] = 'Sensory'
    for n in benefits: node_types[n] = 'Benefit'
    for n in values: node_types[n] = 'Value'

    # レイヤー順に接続: Attr -> Sensory -> Benefit -> Value
    # ※ データによっては一部欠損もあるので、あるものだけでつなぐ
    chain_layers = [l for l in [attrs, senses, benefits, values] if l]

    for i in range(len(chain_layers) - 1):
        source_layer = chain_layers[i]
        target_layer = chain_layers[i+1]
        for u in source_layer:
            for v in target_layer:
                # エッジの重みを加算（頻出するロジックほど強くなる）
                if G.has_edge(u, v):
                    G[u][v]['weight'] += 1
                else:
                    G.add_edge(u, v, weight=1)

print(f"  Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

# ==========================================
# 3. パス探索ロジック (The Strategy)
# ==========================================
def find_winning_story(ocr_text, sensory_text, graph, top_k=3):
    """
    入力キーワード(OCR+Sensory)をStart Nodeとして、
    Graph上のValue Nodeに至る「強いパス」を探す
    """
    # 1. 入力テキストからキーワード化（簡易的な形態素解析代わり）
    # 実際はMeCab等入れた方がいいですが、ここではグラフ内のノードとの文字列マッチで代用
    start_candidates = set()

    # 入力テキスト全体
    full_text = str(ocr_text) + " " + str(sensory_text)

    # グラフにあるノードが、入力テキストに含まれているかチェック
    # (これが一番高速で確実なマッチング)
    for node in graph.nodes():
        if node in full_text:
            # AttributeかSensoryノードならスタート地点に認定
            if node_types.get(node) in ['Attribute', 'Sensory']:
                start_candidates.add(node)

    if not start_candidates:
        return []

    # 2. 経路探索
    found_paths = []

    # ゴール地点（Valueノード）
    value_nodes = [n for n, t in node_types.items() if t == 'Value']

    for start_node in start_candidates:
        # Startから到達可能なValueを探す
        # ※全探索は重いので、深さ制限付きDFSか、ダイクストラを使う
        # ここではNetworkXの便利な関数を使う

        for end_node in value_nodes:
            if nx.has_path(graph, start_node, end_node):
                try:
                    # 重み(weight)を考慮した最短パス（重みの逆数を距離とする）
                    path = nx.shortest_path(graph, source=start_node, target=end_node, weight=None) # シンプルにホップ数で

                    # パスのスコア計算 (エッジの重みの合計)
                    score = 0
                    for i in range(len(path)-1):
                        score += graph[path[i]][path[i+1]]['weight']

                    found_paths.append({
                        "path": path,
                        "score": score,
                        "end_value": end_node
                    })
                except:
                    continue

    # 3. スコア順にソートして上位を返す
    found_paths = sorted(found_paths, key=lambda x: x['score'], reverse=True)

    # パスの重複排除（同じようなパスばかりにならないように）
    unique_paths = []
    seen_ends = set()
    for p in found_paths:
        if p['end_value'] not in seen_ends:
            unique_paths.append(p)
            seen_ends.add(p['end_value'])
        if len(unique_paths) >= top_k:
            break

    return unique_paths

# ==========================================
# 4. 推論実行 (Inference)
# ==========================================
print("🚀 全商品の『勝ち筋』を計算中...")

final_prompts = []

for i in tqdm(range(len(df_input))):
    row = df_input.iloc[i]
    pid = row['ID']
    ocr = row['OCR_TEXT']
    sensory = row['Sensory_Features']

    # パス検索
    paths = find_winning_story(ocr, sensory, G)

    # 結果を整形
    if paths:
        story_lines = []
        for p in paths:
            chain_str = " -> ".join(p['path'])
            story_lines.append(f"- 【訴求軸: {p['end_value']}】: {chain_str}")

        logic_context = "\n".join(story_lines)
        method = "Graph-Guided"
    else:
        # パスが見つからなかった場合のフォールバック（VLMの結果をそのまま使う）
        logic_context = f"特徴: {sensory}\n(グラフ内に一致するパスなしのため、特徴から直接訴求)"
        method = "Direct-Sensory"

    final_prompts.append({
        "ID": pid,
        "OCR": ocr,
        "Sensory": sensory,
        "Logic_Chain": logic_context,
        "Method": method
    })

# ==========================================
# 5. 保存 (プロンプト生成用データ)
# ==========================================
df_prompt = pd.DataFrame(final_prompts)
SAVE_PATH = OUTPUT_DIR / "final_generation_source.csv"
df_prompt.to_csv(SAVE_PATH, index=False)

print(f"\n✅ 推論準備完了！")
print(f"最終生成用の指示データ: {SAVE_PATH}")

# サンプル確認
print("\n【生成されるロジックの例】")
for idx, row in df_prompt.head(3).iterrows():
    print(f"--- ID: {row['ID']} ({row['Method']}) ---")
    print(row['Logic_Chain'])
    print("")

🧠 HVM (Brain) を構築中...
  Nodes: 310, Edges: 464
🚀 全商品の『勝ち筋』を計算中...


100%|██████████| 1190/1190 [00:01<00:00, 615.91it/s]


✅ 推論準備完了！
最終生成用の指示データ: /content/drive/MyDrive/NISHIKA_AI_Competition/output/final_generation_source.csv

【生成されるロジックの例】
--- ID: 06j3BNM0 (Graph-Guided) ---
- 【訴求軸: 安心感】: 食塩 -> まろやかな甘口 -> 安心感

--- ID: 09n0l3iy (Direct-Sensory) ---
特徴: nan
(グラフ内に一致するパスなしのため、特徴から直接訴求)

--- ID: 0B4XRHmr (Graph-Guided) ---
- 【訴求軸: 安心感】: 食塩 -> まろやかな甘口 -> 安心感
- 【訴求軸: おいしさ】: なめらか -> 絶妙なバランス -> おいしさ



PR文を出力する(モデルQwen2.5-32Bの関係でA100じゃないと動かないので、ここだけT4を切って!あと繋げっぱにすんな!)

In [ ]:
# ==========================================
# 0. メモリの緊急掃除 (Cleanup)　ーーA100なら不要かも!!!!!-----
# ==========================================
print("🧹 VLM (画像モデル) を削除してメモリを空けます...")

# 変数が存在するか確認してから削除
try:
    del model
except:
    pass
try:
    del tokenizer
except:
    pass


# 現在のメモリ状況を確認
free_mem = torch.cuda.mem_get_info()[0] / 1e9
print(f"✨ 現在の空きVRAM: {free_mem:.2f} GB")

🧹 VLM (画像モデル) を削除してメモリを空けます...
✨ 現在の空きVRAM: 9.34 GB


In [1]:
import pandas as pd
from tqdm import tqdm
import torch
from unsloth import FastLanguageModel
from pathlib import Path
import re # 正規表現用


# ==========================================
# 1. 設定 & データ読み込み
# ==========================================
PROJECT_ROOT = Path('/content/drive/MyDrive/NISHIKA_AI_Competition')
OUTPUT_DIR = PROJECT_ROOT / "output"

# 指示データ読み込み
INPUT_CSV = OUTPUT_DIR / "final_generation_source.csv"
if not INPUT_CSV.exists():
    raise FileNotFoundError("指示データが見つかりません。")

df_prompt = pd.read_csv(INPUT_CSV)
print(f"📝 最終生成対象: {len(df_prompt)} 件")

# ==========================================
# 2. プロンプト生成関数の定義 (The Brain)
# ==========================================
def generate_pr_copy(row):
    ocr = str(row['OCR']) if not pd.isna(row['OCR']) else ""
    sensory = str(row['Sensory']) if not pd.isna(row['Sensory']) else ""
    logic_chain = str(row['Logic_Chain'])

    # nan対策
    if logic_chain == "nan":
        logic_chain = f"特徴: {sensory} (魅力的な表現で訴求してください)"

    # Gemma-2向けにフォーマットを最適化
    # 松下モデルの定義を「Context」として与えます
    prompt = f"""
あなたはマーケティング心理学に基づいたプロのコピーライターです。
以下の商品情報を元に、消費者の購買意欲を掻き立てるPR文を作成してください。

【適用する理論：松下(1999)の消費者情報処理モデル】
商品は物理的なモノではなく、消費者の心の中で構成される「意味」であるといえます。
以下の3段階のプロセスを文章の中に組み込んでください。
1. **属性的意味 (Attributes)**: 商品が何であるか（事実・スペック）
2. **機能的意味 (Consequences)**: それが何をしてくれるか（利便性・機能）
3. **情緒的意味 (Values)**: それを持つことでどんな気持ちになれるか（幸福感・充足感）

【入力情報】
・テキスト情報(OCR): {ocr}
・視覚的特徴(Sensory): {sensory}

【生成の設計図 (Logic Chain)】
グラフ探索により導き出された、以下の「意味の連鎖」を必ず踏襲してください。
パス: {logic_chain}

【制約条件】
1. 文字数: **45文字以上、85文字以内** (厳守)
2. トーン: ターゲットに寄り添う、共感性の高いトーン。
3. 禁止事項: グラフに含まれない嘘のスペックを捏造しないこと。
4. 出力形式: PR文のみを出力すること。

【出力】
"""
    return prompt

# ==========================================
# 3. 執筆用モデルのロード (model_LLM)
# ==========================================
# Qwen-2.5-32B (4bit量子化) を使う

model_name_llm = "unsloth/Qwen2.5-32B-Instruct-bnb-4bit"

print(f"✍️ 執筆用モデル ({model_name_llm}) をロード中... (model_llm)")

model_llm, tokenizer_llm = FastLanguageModel.from_pretrained(
    model_name = model_name_llm,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model_llm)

# ==========================================
# 4. 生成実行 (model_llm を使用)
# ==========================================
print("🚀 松下モデル × グラフ理論 × {model_name_llm} でPR文を生成中...")

final_results = []

for i in tqdm(range(len(df_prompt))):
    row = df_prompt.iloc[i]

    # 定義した関数を使用
    prompt_text = generate_pr_copy(row)

    # Gemma-2用のチャットテンプレートを適用 (これが精度向上の鍵)
    messages = [
        {"role": "user", "content": prompt_text}
    ]
    inputs = tokenizer_llm.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True, # Assistantのターンから始めさせる
        return_tensors = "pt",
    ).to("cuda")

    try:
        outputs = model_llm.generate(
            input_ids = inputs,
            max_new_tokens = 150,
            use_cache = True,
            temperature = 0.7,
            do_sample = True,
            repetition_penalty = 1.1
        )

        # デコード
        text = tokenizer_llm.decode(outputs[0])

        # クリーニング (Gemma特有のタグ除去)
        # <start_of_turn>model\n ... <end_of_turn> の中身を取り出す
        if "<start_of_turn>model" in text:
            text = text.split("<start_of_turn>model")[-1]

        text = text.replace("<|end_of_text|>", "").replace("<end_of_turn>", "").strip()

        # 余計な「【出力】」などが残っていたら消す
        if "【出力】" in text:
            text = text.split("【出力】")[-1]
        text = text.replace(":", "").replace("：", "").strip()

    except Exception as e:
        print(f"Error at {i}: {e}")
        text = "生成エラー"

    final_results.append({
        "ID": row['ID'],
        "target": text
    })

# ==========================================
# 5. 保存 & 仕上げ
# ==========================================
df_submit = pd.DataFrame(final_results)

# 100文字カット & 文末調整
def final_polish(text):
    text = str(text).strip()
    # 改行を削除
    text = text.replace("\n", "")

    if len(text) > 100:
        # 100文字超えたら、最後の句点で切る
        if "。" in text[:99]:
            return text[:99].rsplit("。", 1)[0] + "。"
        else:
            return text[:100]
    return text

df_submit['target'] = df_submit['target'].apply(final_polish)

SAVE_PATH = OUTPUT_DIR / "submission_theory_guided_final.csv"
df_submit.to_csv(SAVE_PATH, index=False)

print(f"\n✅ 全行程コンプリート！")
print(f"提出用ファイルはこちら: {SAVE_PATH}")
print("\n【生成サンプル (松下モデル適用版)】")
pd.set_option('display.max_colwidth', None)
display(df_submit.head(5))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
📝 最終生成対象: 1190 件
✍️ 執筆用モデル (unsloth/Qwen2.5-32B-Instruct-bnb-4bit) をロード中... (model_llm)
==((====))==  Unsloth 2025.12.5: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

🚀 松下モデル × グラフ理論 × {model_name_llm} でPR文を生成中...


100%|██████████| 1190/1190 [1:19:53<00:00,  4.03s/it]


✅ 全行程コンプリート！
提出用ファイルはこちら: /content/drive/MyDrive/NISHIKA_AI_Competition/output/submission_theory_guided_final.csv

【生成サンプル (松下モデル適用版)】


,ID,target
0,06j3BNM0,<|im_end|><|im_start|>assistant安心の素材とシンプルな味わい。子どもへの愛情を込めた一粒ずつ、まろやかな甘口でお子様も安心して楽しめるおむすびです。<|im_end|>
1,09n0l3iy,<|im_end|><|im_start|>assistant香辛料の豊かな風味で日常を彩るアジアングリーンカレー。家庭で手軽に本格料理を楽しもう。幸せな食卓へ。<|im_end|>
2,0B4XRHmr,<|im_end|><|im_start|>assistant国産米100%と和三盆糖を使用した絶妙なバランス。滑らかな舌触りで、安心しておいしくいただけます。<|im_end|>
3,0CIltEix,<|im_end|><|im_start|>assistant透明な爽快シールで、マスク生活もすっきり。メントールの涼しさで気分リフレッシュ。日本製、10枚入り。<|im_end|>
4,0CupiBOe,<|im_end|><|im_start|>assistantチョコフレークの絶妙なバランスが溶け出すなめらかさとザクザクとした食感で、手軽に満足感を得られます。


In [8]:
import pandas as pd
import re
from pathlib import Path

# ==========================================
# 設定　クリーンンぐ
# ==========================================
PROJECT_ROOT = Path('/content/drive/MyDrive/NISHIKA_AI_Competition')
OUTPUT_DIR = PROJECT_ROOT / "output"

# 生成された生のファイル
INPUT_FILE = OUTPUT_DIR / "submission_theory_guided_final.csv"
# 提出用ファイル（完成品）
OUTPUT_FILE = OUTPUT_DIR / "submission_clean.csv"

# ==========================================
# クリーニング処理
# ==========================================
df = pd.read_csv(INPUT_FILE)

def clean_generated_text(text):
    if pd.isna(text):
        return ""
    text = str(text)

    # 1. LLM特有のタグを除去 (Gemma/Qwenなどが残すゴミ)
    # <|im_start|>, <|im_end|>, <|endoftext|> などを正規表現で一掃
    text = re.sub(r'<\|.*?\|>', '', text)

    # 2. 不要なロール名の除去
    text = text.replace("assistant", "").replace("model", "")

    # 3. 文頭・文末の空白と改行を除去
    text = text.replace("\n", "").strip()

    # 4. ダブルクォートなどの調整（CSV破損防止）
    text = text.replace('"', '')

    return text

print(f"🧹 クリーニング前: {len(df)} 件")

# 実行
df['target'] = df['target'].apply(clean_generated_text)

# コンペの提出フォーマットはおそらく 'ID' と 'label' カラムが一般的です
# 必要に応じて列名を変更します
df = df.rename(columns={'target': 'label'})

# 最終確認: 空のデータがないか
empty_count = len(df[df['label'] == ""])
if empty_count > 0:
    print(f"⚠️ 注意: 空のテキストが {empty_count} 件あります！")
    # 空の場合は「魅力的な商品です。」などで埋める救済措置
    df.loc[df['label'] == "", 'label'] = "日々の生活を豊かにする、魅力あふれる一品です。"

# 保存
df[['ID', 'label']].to_csv(OUTPUT_FILE, index=False)

print(f"✨ クリーニング完了！")
print(f"提出用ファイル: {OUTPUT_FILE}")
print("\n【最終仕上がりサンプル】")
print(df.head())

🧹 クリーニング前: 1190 件
✨ クリーニング完了！
提出用ファイル: /content/drive/MyDrive/NISHIKA_AI_Competition/output/submission_clean.csv

【最終仕上がりサンプル】
         ID                                              label
0  06j3BNM0  安心の素材とシンプルな味わい。子どもへの愛情を込めた一粒ずつ、まろやかな甘口でお子様も安心し...
1  09n0l3iy  香辛料の豊かな風味で日常を彩るアジアングリーンカレー。家庭で手軽に本格料理を楽しもう。幸せな...
2  0B4XRHmr  国産米100%と和三盆糖を使用した絶妙なバランス。滑らかな舌触りで、安心しておいしくいただけます。
3  0CIltEix  透明な爽快シールで、マスク生活もすっきり。メントールの涼しさで気分リフレッシュ。日本製、10...
4  0CupiBOe  チョコフレークの絶妙なバランスが溶け出すなめらかさとザクザクとした食感で、手軽に満足感を得ら...


In [10]:
import pandas as pd
from pathlib import Path

# 設定
PROJECT_ROOT = Path('/content/drive/MyDrive/NISHIKA_AI_Competition')
OUTPUT_DIR = PROJECT_ROOT / "output"
SUBMISSION_FILE = OUTPUT_DIR / "submission_clean.csv" # さっきのファイル

# 読み込み
df = pd.read_csv(SUBMISSION_FILE)

print("🔍 提出ファイル検証スタート...")

# 1. カラム名の統一 (修正済み)
# 'label' があれば 'target' に変更する
if 'label' in df.columns:
    df = df.rename(columns={'label': 'target'})

# 2. 全行チェック
expected_rows = 1190
if len(df) != expected_rows:
    print(f"❌ 行数エラー: {len(df)} (正解: {expected_rows})")
else:
    print(f"✅ 行数OK: {len(df)}")

# 3. 文字数チェック (targetカラムを使う)
df['length'] = df['target'].astype(str).str.len()
long_rows = df[df['length'] > 100]
short_rows = df[df['length'] < 20]

if len(long_rows) > 0:
    print(f"❌ 100文字超えが {len(long_rows)} 件あります！提出不可の可能性あり。")
    print(long_rows[['ID', 'target', 'length']])
else:
    print(f"✅ 文字数上限OK (最大 {df['length'].max()} 文字)")

if len(short_rows) > 0:
    print(f"⚠️ 20文字未満が {len(short_rows)} 件あります。少し短いかも？")
    print(short_rows[['ID', 'target', 'length']])
else:
    print(f"✅ 文字数下限OK (最小 {df['length'].min()} 文字)")

# 4. 欠損チェック
if df['target'].isnull().any() or (df['target'] == "").any():
    print("❌ 空のラベルが含まれています！")
else:
    print("✅ 欠損なしOK")

# 保存 (上書きまたは別名保存)
FINAL_FILE = OUTPUT_DIR / "submission_final_checked.csv"
df[['ID', 'target']].to_csv(FINAL_FILE, index=False)

print(f"\n🎉 検証完了！提出用ファイルを作成しました: {FINAL_FILE}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/NISHIKA_AI_Competition/output/submission_clean.csv'

参照

日高モデル

https://www.jaist.ac.jp/~shhidaka/cv_publications/Hidaka2013IEICE.pdf

消費者情 報処理 モデル による 「意 味」 の概 念分析(松下モデル)

https://www.jstage.jst.go.jp/article/acs1993/6/2/6_2_29/_pdf/-char/ja

A Dynamics for Advertising on Networks

https://theory.epfl.ch/vishnoi/Publications_files/CDVWine17.pdf

Celis

https://www.cs.yale.edu/homes/vishnoi/Polarization.pdf